In [1]:
import duckdb
from pathlib import Path

# 1. Robust Project Root Discovery
# Looks for the unique folder name of your repo
def get_project_root(target_name="triangle-spatial-data-engine"):
    for parent in Path.cwd().parents:
        if parent.name == target_name:
            return parent
    return Path.cwd() # Fallback to current dir

root = get_project_root()
db_path = root / "data" / "triangle_engine.db"

# 2. Connect & Initialize Spatial
# Using a context manager or persistent connection
con = duckdb.connect(str(db_path))
con.execute("INSTALL spatial; LOAD spatial;")

print(f"Engine Connected: {db_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Engine Connected: /workspaces/triangle-spatial-data-engine/data/triangle_engine.db


In [2]:
# 1. Project-specific paths (using Path objects)
proj_dir = root / "projects" / "01_food_desert_transit"
bronze_dir = proj_dir / "data" / "bronze"
silver_dir = proj_dir / "data" / "silver"

# 2. Ensure directories exist (Professional safeguard)
silver_dir.mkdir(parents=True, exist_ok=True)

print(f"Project 01 Context Set.\nBronze: {bronze_dir}")

Project 01 Context Set.
Bronze: /workspaces/triangle-spatial-data-engine/projects/01_food_desert_transit/data/bronze


In [3]:
# 1. Define specific source files
bus_stops_file = bronze_dir / "bus_stops.parquet"
grocery_file = bronze_dir / "grocery_stores.parquet"

# 2. Mount Views with absolute paths (No wildcards)
con.execute(f"CREATE OR REPLACE VIEW bus_stops AS SELECT * FROM '{bus_stops_file.as_posix()}'")
con.execute(f"CREATE OR REPLACE VIEW grocery_stores AS SELECT * FROM '{grocery_file.as_posix()}'")

print("Views isolated and mounted.")

Views isolated and mounted.


In [4]:
# Audit the structure of bus stops
print("--- Bus Stops Schema ---")
con.execute("DESCRIBE bus_stops;").pl()

--- Bus Stops Schema ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""OBJECTID""","""INTEGER""","""YES""",null,null,null
"""Stop_Code""","""INTEGER""","""YES""",null,null,null
"""Stop_ID""","""INTEGER""","""YES""",null,null,null
"""Stop_Name""","""VARCHAR""","""YES""",null,null,null
"""Street""","""VARCHAR""","""YES""",null,null,null
…,…,…,…,…,…
"""Construction_Set""","""VARCHAR""","""YES""",null,null,null
"""Bench_Install_Year""","""VARCHAR""","""YES""",null,null,null
"""SWS_Pickup""","""VARCHAR""","""YES""",null,null,null


In [5]:
# Audit the structure of Grocery Stores
print("--- Grocery Stores Schema ---")
con.execute("DESCRIBE grocery_stores;").pl()

--- Grocery Stores Schema ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""type""","""VARCHAR""","""YES""",null,null,null
"""id""","""BIGINT""","""YES""",null,null,null
"""lat""","""DOUBLE""","""YES""",null,null,null
"""lon""","""DOUBLE""","""YES""",null,null,null
"""tags""","""STRUCT(""addr:city"" VARCHAR, ""a…","""YES""",null,null,null
…,…,…,…,…,…
"""diet:organic""","""VARCHAR""","""YES""",null,null,null
"""diet:seafood""","""VARCHAR""","""YES""",null,null,null
"""diet:vegan""","""VARCHAR""","""YES""",null,null,null


### Data Exploration & Validation Strategy

Before performing any transformations or moving data from bronze to silver, I am conducting an audit of the raw files. A schema definition only shows the shape of the data; it does not reveal the health or the actual content.

**Approach:**

* **Individualized Inspection:** Splitting the audit into separate cells to isolate outputs and avoid information overload.
* **Bus Stop Integrity:** Verifying the legitimacy of the spatial points. I need to confirm the record counts and ensure the dataset isn't riddled with anomalies that would break future analysis.
* **Grocery Store Deconstruction:** With 72 columns, most of this dataset is noise. I need to explore the nested tags structure to identify which attributes are actually populated and relevant.

In [6]:
columns = con.table("bus_stops").columns
cols_per_row = 4

for i in range(0, len(columns), cols_per_row):
    row = columns[i:i + cols_per_row]
    print("".join(f"{name:<30}" for name in row))

OBJECTID                      Stop_Code                     Stop_ID                       Stop_Name                     
Street                        Cross_Street                  Position                      Stop_Lat                      
Stop_Lon                      Downtown                      Council_District              Sign_Crew                     
Road_Maint                    GoRaleigh                     GoTriangle                    Wolfline                      
Wake_Forest                   GoDurham                      GoCary                        Shelter                       
Shelter_Type                  Shelter_Count                 Bench                         Bench_Type                    
Bench_Count                   Trash_Can                     Trash_Can_Type                Trash_Can_Count               
Landing_Pad                   Amenity_Pad                   Pad_Size                      Bike_Rack                     
Lighting                      Li

In [7]:
# Simple list comprehension for quick reference
print(con.table("grocery_stores").columns)

['type', 'id', 'lat', 'lon', 'tags', 'brand', 'brand:wikidata', 'name', 'shop', 'geometry', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:state', 'addr:street', 'wheelchair', 'branch', 'opening_hours', 'phone', 'ref', 'website', 'addr:unit', 'addr:country', 'origin', 'source', 'check_date', 'check_date:opening_hours', 'butcher', 'contact:email', 'contact:phone', 'contact:website', 'cuisine', 'addr:county', 'operator', 'payment:american_express', 'payment:apple_pay', 'payment:cheque', 'payment:coins', 'payment:debit_cards', 'payment:discover_card', 'payment:ebt', 'payment:google_pay', 'payment:mastercard', 'payment:notes', 'payment:snap', 'payment:visa', 'payment:wic', 'operator:wikidata', 'payment:app:walmart', 'payment:samsung_pay', 'ref:walmart', 'sensory_friendly:accommodation', 'sensory_friendly:conditional', 'name:en', 'wikidata', 'wikipedia', 'organic', 'atm', 'internet_access', 'payment:cash', 'payment:credit_cards', 'payment:money_order', 'diet:halal', 'diet:kosher', 

In [8]:
# Bus stop integrity & distribution audit
con.execute("""
    SELECT 
        count(*) as total_records,
        count(geometry) as records_with_geometry,
        count(*) filter (where Stop_Name IS NULL OR Stop_Name = '') as missing_names,
        count(*) filter (where Stop_Lat = 0 OR Stop_Lat IS NULL) as null_island_check
    FROM bus_stops;
""").pl()

total_records,records_with_geometry,missing_names,null_island_check
i64,i64,i64,i64
1402,1402,0,113


In [9]:
# Grocery integrity audit
con.execute("""
    SELECT 
        COUNT(*) AS total_records,
        COUNT(*) FILTER (WHERE lat != 0 AND lon != 0) AS has_valid_coords,
        COUNT(*) FILTER (WHERE lat = 0 OR lon = 0) AS null_island_records,
        COUNT(*) FILTER (WHERE shop = 'supermarket') AS supermarket_count,
        COUNT(*) FILTER (WHERE shop = 'grocery') AS grocery_count,
        COUNT(*) FILTER (WHERE name IS NULL OR name = '') AS unnamed_stores
    FROM grocery_stores
    WHERE shop IN ('supermarket', 'grocery');
""").pl()

total_records,has_valid_coords,null_island_records,supermarket_count,grocery_count,unnamed_stores
i64,i64,i64,i64,i64,i64
67,67,0,67,0,0


In [10]:
# Inspect bus stop geometry sample
print("--- Bus Stops Metadata (Geometry Sample) ---")
con.execute("""
    SELECT 
        ST_AsText(geometry) AS sample_wkt_geometry 
    FROM bus_stops 
    LIMIT 3
""").pl()

--- Bus Stops Metadata (Geometry Sample) ---


sample_wkt_geometry
str
"""POINT (-78.6533558284032 35.78…"
"""POINT (-78.6383530001644 35.77…"
"""POINT (-78.6403999992872 35.73…"


In [11]:
# Check grocery coordinate ranges 
print("--- Grocery Coordinate Ranges ---")
con.execute("""
    SELECT 
        min(lat) AS min_lat, 
        max(lat) AS max_lat, 
        min(lon) AS min_lon, 
        max(lon) AS max_lon 
    FROM grocery_stores;
""").pl()

--- Grocery Coordinate Ranges ---


min_lat,max_lat,min_lon,max_lon
f64,f64,f64,f64
35.702224,35.899037,-78.792064,-78.505275


In [12]:
# Bus stop geometry check 
print("--- Bus Stop Geometry Sample ---")
con.execute("SELECT ST_AsText(geometry) AS geom_sample FROM bus_stops LIMIT 1").pl()

--- Bus Stop Geometry Sample ---


geom_sample
str
"""POINT (-78.6533558284032 35.78…"


In [13]:
# Grocery coordinate check
print("\n--- Grocery Coordinate Sample ---")
con.execute("SELECT lon, lat FROM grocery_stores LIMIT 1").pl()


--- Grocery Coordinate Sample ---


lon,lat
f64,f64
-78.66068,35.886024


In [14]:
# Preview with explicit WKT casting to avoid the Internal Arrow Error
print("--- Silver Bus Stops (Preview) ---")
con.execute("""
    SELECT 
        stop_id, 
        stop_name, 
        ST_AsText(geometry) as geom_wkt 
    FROM silver_bus_stops 
    LIMIT 3
""").pl()

--- Silver Bus Stops (Preview) ---


stop_id,stop_name,geom_wkt
str,str,str
"""1201""","""Hillsborough St at Mayo St""","""POINT (-78.6533558284032 35.78…"
"""1284""","""Wilmington St at Cabarrus St""","""POINT (-78.6383530001644 35.77…"
"""1539""","""Hammond Rd at Chapanoke Rd (SE…","""POINT (-78.6403999992872 35.73…"


In [15]:
# Preview with explicit WKT casting to avoid the Internal Arrow Error

print("\n--- Silver Grocery Stores (Preview) ---")
con.execute("""
    SELECT 
        grocery_id, 
        grocery_name, 
        shop_type, 
        ST_AsText(geometry) as geom_wkt 
    FROM silver_grocery_stores 
    LIMIT 3
""").pl()


--- Silver Grocery Stores (Preview) ---


grocery_id,grocery_name,shop_type,geom_wkt
str,str,str,str
"""729687624""","""Food Lion""","""supermarket""","""POINT (-78.6606799 35.8860239)"""
"""821354490""","""Food Lion""","""supermarket""","""POINT (-78.5896006 35.8235449)"""
"""821995253""","""Food Lion""","""supermarket""","""POINT (-78.5787629 35.8453896)"""


In [16]:
# Identity & brand integrity audit
print("--- Duplicate Bus Stop IDs (Expecting 0) ---")
con.execute("""
    SELECT stop_id, count(*) as occurrence_count
    FROM bus_stops 
    GROUP BY stop_id 
    HAVING count(*) > 1;
""").pl()

--- Duplicate Bus Stop IDs (Expecting 0) ---


Stop_ID,occurrence_count
i32,i64
9464,2
8858,2


In [17]:
# Identity & brand integrity audit
print("\n--- Unnamed & Brand Distribution (Grocery) ---")
con.execute("""
    SELECT 
        count(*) filter (where name IS NULL OR name = '') as missing_names,
        count(*) filter (where brand IS NULL) as unbranded_stores,
        -- Check for 'Wholesale' or 'Club' in name/brand to flag for 15-min walk exclusion
        count(*) filter (where name ILIKE '%Wholesale%' OR brand ILIKE '%Costco%' OR brand ILIKE '%BJ%') as wholesale_clubs
    FROM grocery_stores;
""").pl()


--- Unnamed & Brand Distribution (Grocery) ---


missing_names,unbranded_stores,wholesale_clubs
i64,i64,i64
0,17,0


In [18]:
# Identity & brand integrity audit
print("\n--- Top 10 Grocery Brands ---")
con.execute("""
    SELECT brand, count(*) as store_count 
    FROM grocery_stores 
    WHERE brand IS NOT NULL
    GROUP BY brand 
    ORDER BY store_count DESC 
    LIMIT 10;
""").pl()


--- Top 10 Grocery Brands ---


brand,store_count
str,i64
"""Food Lion""",20
"""Harris Teeter""",8
"""ALDI""",4
"""Walmart""",3
"""Lowes Foods""",3
"""Lidl""",2
"""The Fresh Market""",2
"""Whole Foods Market""",2
"""Trader Joe's""",2


In [19]:
# Check stop_ID, name, and geometry
con.execute("""
    SELECT 
        Stop_ID, 
        Stop_Name, 
        Status, 
        ST_AsText(geometry) AS geom_wkt,
        -- Create a hash/fingerprint to quickly see if the records are 100% identical
        md5(concat(Stop_ID, Stop_Name, ST_AsText(geometry))) AS record_fingerprint
    FROM bus_stops 
    WHERE Stop_ID IN ('8858', '9464')
    ORDER BY Stop_ID, record_fingerprint;
""").pl()

Stop_ID,Stop_Name,Status,geom_wkt,record_fingerprint
i32,str,str,str,str
8858,"""Raleigh Blvd at Glascock St (I…","""Active""","""POINT (-78.6132051288152 35.79…","""384edfd98ae57cbf38454a3680af84…"
8858,"""Oberlin Rd at Van Dyke Ave""","""Active""","""POINT (-78.6609230000997 35.79…","""be4de9f7697f8d9d5b119b62dc2eae…"
9464,"""E Lane St at Heck St""","""Active""","""POINT (-78.6243463065577 35.78…","""4ca462df77b4cd625234d4347220f2…"
9464,"""E Lane St. at Heck St.""","""Active""","""POINT (-78.6244151758989 35.78…","""ad47a75897282f970a048017bdb247…"


In [20]:
# Investigating none brands
print("--- Unbranded Grocery Store Sample ---")
con.execute("""
    SELECT 
        name, 
        shop AS shop_type, 
        brand 
    FROM grocery_stores 
    WHERE brand IS NULL 
      AND shop IN ('supermarket', 'grocery')
      AND (name IS NOT NULL AND name != '')
    LIMIT 10;
""").pl()

--- Unbranded Grocery Store Sample ---


name,shop_type,brand
str,str,str
"""Compare""","""supermarket""",null
"""Grand Asia Market""","""supermarket""",null
"""Casablanca Market""","""supermarket""",null
"""Harmony Farms""","""supermarket""",null
"""Carolina Open Air Market""","""supermarket""",null
"""Harmony Mediterranean Market""","""supermarket""",null
"""Weaver Street Market""","""supermarket""",null
"""El Mandado Supermarket""","""supermarket""",null
"""NC International Grocery Store""","""supermarket""",null


In [21]:
# Integrity check. Are any stores missing both name and brand?
print("\n--- Critical Missing Data (No Name AND No Brand) ---")
con.execute("""
    SELECT count(*) as total_anonymous_stores
    FROM grocery_stores
    WHERE (name IS NULL OR name = '') 
      AND (brand IS NULL OR brand = '');
""").pl()


--- Critical Missing Data (No Name AND No Brand) ---


total_anonymous_stores
i64
0


In [22]:
# 1. Spatial drift audit (ID collisions with different coordinates)
print("--- Spatial Drift: IDs with Multiple Locations ---")
con.execute("""
    WITH drift_analysis AS (
        SELECT 
            Stop_ID, 
            COUNT(DISTINCT ST_AsText(geometry)) as location_count,
            -- Calculate the distance between the drifting points (if they are close)
            ST_Distance(MIN(geometry), MAX(geometry)) as drift_distance_ft
        FROM bus_stops 
        GROUP BY Stop_ID 
        HAVING location_count > 1
    )
    SELECT 
        Stop_ID, 
        location_count,
        drift_distance_ft
    FROM drift_analysis;
""").pl()

--- Spatial Drift: IDs with Multiple Locations ---


Stop_ID,location_count,drift_distance_ft
i32,i64,f64
8858,2,0.04779
9464,2,0.000139


In [23]:
# Summary count
print("\n--- Total IDs Affected by Spatial Drift ---")
con.execute("""
    SELECT count(*) as total_drifting_ids 
    FROM (
        SELECT Stop_ID FROM bus_stops 
        GROUP BY Stop_ID HAVING COUNT(DISTINCT ST_AsText(geometry)) > 1
    )
""").pl()


--- Total IDs Affected by Spatial Drift ---


total_drifting_ids
i64
2


In [24]:
# Final laboratory test: using list() to feed ST_Collect
con.execute("""
    SELECT 
        Stop_ID, 
        TRIM(REPLACE(Stop_Name, '.', '')) as clean_name,
        COUNT(*) as records_merged,
        ST_AsText(ST_Centroid(ST_Collect(list(geometry)))) as final_geom
    FROM bus_stops 
    WHERE Stop_ID IN ('8858', '9464')
    GROUP BY Stop_ID, clean_name
    ORDER BY Stop_ID;
""").pl()

Stop_ID,clean_name,records_merged,final_geom
i32,str,i64,str
8858,"""Raleigh Blvd at Glascock St (I…",1,"""POINT (-78.6132051288152 35.79…"
8858,"""Oberlin Rd at Van Dyke Ave""",1,"""POINT (-78.6609230000997 35.79…"
9464,"""E Lane St at Heck St""",2,"""POINT (-78.6243807412283 35.78…"


In [25]:
# Final grocery audit: non-supermarkets hiding in our grocery list?
con.execute("""
    SELECT 
        shop, 
        COUNT(*) as store_count
    FROM grocery_stores 
    GROUP BY shop 
    ORDER BY store_count DESC;
""").pl()

shop,store_count
str,i64
"""supermarket""",67
